In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload

In [2]:
%autoreload
import torch
import pandas as pd
import torch.nn as nn
from pytorch_pretrained_bert import BertTokenizer, BertAdam
from torch.utils.data import DataLoader
from torch.nn import BCEWithLogitsLoss, MSELoss
from torch.optim import Adam
from source.bert.data import Data
from source.bert.model import Bert
from source.bert.train import trainModel

In [3]:
def loadModel(model, path):
    results = torch.load(path)
    model.load_state_dict(results['model_state_dict'])
    loss = results['loss']
    print('Model Loaded:', 'Loss:', loss)
    return model, loss

In [4]:
def customLoss(preds, label, aux, aux_label, weight):
    mse_loss = MSELoss(reduction='none')(preds.squeeze(), label.squeeze())
    mse_loss = torch.mul(weight, mse_loss)
    aux_loss = MSELoss(reduction='none')(aux.squeeze(), aux_label.squeeze())
    tot_loss = 3.5 * mse_loss.mean() + aux_loss.mean()
    return tot_loss

In [ ]:
def train_model(device, fold):
    train_data = pd.read_csv('../../data/train_data_{}.csv'.format(fold))
    valid_data = pd.read_csv('../../data/valid_data_{}.csv'.format(fold))
    print('Dataset:',train_data.shape, valid_data.shape)
    params = {}
    params['pretrained_model_name_or_path'] = 'bert-base-uncased'
    params['do_lower_case'] = True
    params['never_split'] = ("[UNK]", "[SEP]", "[PAD]", "[CLS]", "[MASK]")
    tokenizer = BertTokenizer.from_pretrained(**params)
    train_data = Data(tokenizer, train_data)
    valid_data = Data(tokenizer, valid_data)
    params = {}
    params['num_workers'] = 6
    params['pin_memory'] = True
    params['drop_last'] = True
    train_loader = DataLoader(dataset=train_data, batch_size=12, shuffle=True, **params)
    valid_loader = DataLoader(dataset=valid_data, batch_size=8, shuffle=False, **params)
    model = Bert().float().to(device)
    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
    param_optimizer = list(model.named_parameters())
    optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = BertAdam(optimizer_grouped_parameters, lr=2e-5, warmup=0.05, t_total=25000)
    params = {}
    params['train'] = train_loader
    params['valid'] = valid_loader
    params['device'] = device
    params['model'] = model
    params['optimizer'] = optimizer
    params['loss_fn'] = customLoss
    params['save'] = '../../model/bert/fold-{}/'.format(fold)
    params['batch'] = 12
    trainModel(**params)
    return None

In [ ]:
train_model('cuda:0', 1)

Dataset: (1443898, 9) (360976, 9)


  0%|          | 5184/1443888 [01:39<7:32:01, 53.05it/s, train_loss=0.5829, valid_loss=-0.0100]

In [ ]:
train_model('cuda:0', 2)